# Robust Online Inertia Estimation Method
Adapted from Liu et al. "On-Line Inertia Estimation for Synchronous and Non-Synchronous Devices", specifically their E2 method, which takes into account the controller's damping and primary frequency control. This helps to remove oscillations in their estimated inertia. 

In [1]:
%matplotlib inline

from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import andes

andes.config_logger(stream_level=30)

In [2]:
# path setup
from pathlib import Path
project_dir = Path.cwd()
test_case_dir = project_dir / "test_cases" # all test case excel sheets
disturbance_profile_dir = project_dir / "disturbance_profiles" # disturbance profiles for ANDES perturbation
out_dir = project_dir / "out"

case_path = andes.get_case(test_case_dir / "regf2_testbench.xlsx")
STEP_TIME = 0.01 # 10 ms

## Load in case file and, run power flow and TDS

In [3]:
ss = andes.load(case_path, setup=False)
ss.REGF2.set("Tr",1,0.005)
print(ss.REGF2.Tr.v)

ss.setup()
ss.config.warn_abnormal = 0
ss.PFlow.run()

ss.TDS.config.tf = 20 # Baruzzi et al ran simulation for 9s after disturbance
ss.TDS.config.fixt = 1 # fixed step size
ss.TDS.config.tstep = STEP_TIME # matches Baruzzi et al. 

ss.TDS.config.no_tqdm = 1
ss.TDS.config.criteria = 1
ss.TDS.run()

[0.005]


True

In [4]:
# get frequency, RoCoF, and power time-series. 
tds_df = pd.concat([ss.TDS.get_timeseries(ss.BusROCOF.f), 
                    ss.TDS.get_timeseries(ss.BusROCOF.Wf_y), 
                    ss.TDS.get_timeseries(ss.REGF2.Pe)],
                    axis=1)

tds_df.columns = ["f", "df/dt", "p"]

tds_df

,f,df/dt,p
0.0000,1.000000,0.000000e+00,0.200000
0.0001,1.000000,0.000000e+00,0.200000
0.0101,1.000000,0.000000e+00,0.200000
0.0201,1.000000,0.000000e+00,0.200000
0.0301,1.000000,1.301203e-10,0.200000
...,...,...,...
19.9601,0.999999,-9.415564e-09,0.200011
19.9701,0.999999,-9.415564e-09,0.200011
19.9801,0.999999,-9.415564e-09,0.200011
19.9901,0.999999,-9.415564e-09,0.200011


## Inertia Calculation

In [5]:
t0 = 0
tf = 20

# these are taken from Liu et al.
#TM = 0.001
#TD = 0.001 
TM = 0.1
TD = 10e-4

eps_x = 10e-5
eps_w = 10e-5

Kp = 50
Ki = 1
Tf = 0.0001

In [6]:
# calculate RoCoF, derivative of RoCoF (RoRoCOF?? lol), and derivative of power (RoCoP)
# need to interpolate between values b/c continuous-- called by solver

domega_dt = CubicSpline(tds_df.index, tds_df.loc[:,"df/dt"])      
d2omega_dt2 = domega_dt.derivative()               
dp_dt = CubicSpline(tds_df.index, tds_df.loc[:,"p"]).derivative()

In [7]:
def gamma(x, eps):
    return np.where(x >= eps, -1.0, np.where(x <= -eps, 1.0, 0.0))

In [8]:
def rhs_f(t, y, TM, TD, eps_x, eps_w, Kp, Ki, Tf):
    Mstar, Dstar, domega_int, dp_int, omega_star, d_omega_star = y

    """
    dp = dp_dt(t)
    dw_raw = domega_dt(t)
    e = dw_raw - omega_star

     d_omega_star_dt = Ki * Kp * e
    d_d_omega_star_dt = (Kp * e - 1 * d_omega_star) / Tf   

    dw = omega_star
    d2w = d_omega_star
    """

    dp = dp_dt(t)
    d2w = d2omega_dt2(t)
    dw = domega_dt(t)

    print(f"t = {t}| dw = {dw}| d2w = {d2w} | dp = {dp} | Mstar = {Mstar}")

    dMstar = gamma(d2w, eps_x) * (dp - Mstar*d2w - Dstar*dw) / TM
    dDstar = gamma(domega_int, eps_w) * (dp_int - Mstar*dw - Dstar*domega_int) / TD
    ddomega_int = dw     
    ddp_int = dp     

    #print(f"{t}: dMstar {dMstar}")    

    return [dMstar, dDstar, ddomega_int, ddp_int, d_omega_star_dt, d_d_omega_star_dt]

sol = solve_ivp(rhs, [t0, tf], y0=[0, 0, 0, 0, 0, 0],
                 args=(TM, TD, eps_x, eps_w, Kp, Ki, Tf),
                 method='Radau', dense_output=True,
                 t_eval=tds_df.index)

NameError: name 'rhs' is not defined

In [ ]:
def rhs(t, y, TM, TD, eps_x, eps_w):
    Mstar, Dstar, domega_int, dp_int = y

    dp = dp_dt(t)
    d2w = d2omega_dt2(t)
    dw = domega_dt(t)

    dMstar = gamma(d2w, eps_x) * (dp - Mstar*d2w - Dstar*dw) / TM
    dDstar = gamma(domega_int, eps_w) * (dp_int - Mstar*dw - Dstar*domega_int) / TD
    ddomega_int = dw     
    ddp_int = dp     

    #print(f"{t}: dMstar {dMstar}")    

    return [dMstar, dDstar, ddomega_int, ddp_int]

In [ ]:
sol = solve_ivp(rhs, [t0, tf], y0=[0, 0, 0, 0],
                 args=(TM, TD, eps_x, eps_w),
                 method='Radau', dense_output=True,
                 t_eval=tds_df.index)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sol.t, sol.y[0])
ax.axline((0, 5), slope=0, color="black", linestyle=":")
#ax.set_ylim(0, 5.5)

In [ ]:
ss.TDS.plot(ss.REGCV2.delta)

In [ ]:
plt.plot(tds_df.index, tds_df.loc[:,"p"])